# Fairlearn: is the model fair, to whom, and what can we do about it?

The DVC and MLflow lessons kept showing one uncomfortable number: the model catches
**fewer malnourished girls than boys**. This notebook asks properly:

1. Who gets flagged, and who gets missed?
2. Four definitions of "fair": which would you choose for girls and boys, and for regions?
3. If we choose equal opportunity: what it changes, and what it costs.
4. Decide, document, monitor.

**Setting.** A clinic can follow up 20% of visits. The model ranks children by risk and
the top 20% are flagged. The costly error is a **miss**: a child who becomes acutely
malnourished but wasn't flagged. A false alarm costs a follow-up visit.

Fairlearn's core tool is **`MetricFrame`**: any metric, computed overall *and* per group.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, "../03-mlflow")       # reuse the MLflow lesson's data/model/evaluation helpers
import emr_model as em

import numpy as np
import pandas as pd
from fairlearn.metrics import (MetricFrame, equal_opportunity_difference, false_negative_rate,
                               selection_rate, true_positive_rate)
from sklearn.metrics import precision_score, roc_auc_score

pd.set_option("display.width", 160, "display.precision", 3)
RAW = Path("data/raw")                   # this lesson's own data folder

# Train on 2024 (as in the other lessons), evaluate on a full YEAR after deployment (2025):
# 24,000 visits, so small groups still have enough cases to say something.
em.simulate("2024-01", 12, out=RAW)
em.simulate("2025-01", 12, out=RAW)
train = em.load_months("2024-01", "2024-12", raw=RAW)
evalset = em.load_months("2025-01", "2025-12", raw=RAW)

model = em.build_model("logistic").fit(train[em.FEATURES], train[em.LABEL])
y = evalset[em.LABEL].to_numpy()                                  # what happened
score = model.predict_proba(evalset[em.FEATURES])[:, 1]           # predicted risk
flag = (score >= np.quantile(score, 0.8)).astype(int)             # top 20% flagged
print(f"{len(evalset)} visits, {y.mean():.1%} developed GAM; ROC-AUC {roc_auc_score(y, score):.3f}; "
      f"flagged {flag.mean():.0%}; sensitivity {flag[y == 1].mean():.3f}")

24000 visits, 11.0% developed GAM; ROC-AUC 0.814; flagged 20%; sensitivity 0.577


## 1. Who gets flagged, and who gets missed?

In [2]:
metrics = {
    "prevalence":  lambda yt, yp: yt.mean(),      # share who developed GAM
    "alert_rate":  selection_rate,                # share flagged
    "sensitivity": true_positive_rate,            # of those who developed GAM, share flagged
    "miss_rate":   false_negative_rate,           # of those who developed GAM, share MISSED
    "ppv":         precision_score,               # of those flagged, share who developed GAM
    "n":           lambda yt, yp: len(yt),
}
by_sex = MetricFrame(metrics=metrics, y_true=y, y_pred=flag, sensitive_features=evalset["sex"])
print("overall:", by_sex.overall.round(3).to_dict())
by_sex.by_group

overall: {'prevalence': 0.11, 'alert_rate': 0.2, 'sensitivity': 0.577, 'miss_rate': 0.423, 'ppv': 0.319, 'n': 24000.0}


,prevalence,alert_rate,sensitivity,miss_rate,ppv,n
sex,,,,,,
F,0.111,0.191,0.541,0.459,0.317,11685.0
M,0.109,0.209,0.612,0.388,0.321,12315.0


In [3]:
print("largest gap between groups (difference):")
print(by_sex.difference().round(3).to_dict())
print("equal_opportunity_difference:", round(equal_opportunity_difference(y, flag, sensitive_features=evalset["sex"]), 3))

largest gap between groups (difference):
{'prevalence': 0.002, 'alert_rate': 0.019, 'sensitivity': 0.071, 'miss_rate': 0.071, 'ppv': 0.004, 'n': 630.0}


equal_opportunity_difference: 0.071


**Read it like a clinician.** In 2025, girls and boys develop malnutrition at about the same
rate (~11%), but of the children who do, the model flags **54% of girls vs 61% of boys**.
Out of every 100 malnourished girls, about 7 more are missed than among boys. The
flags are equally reliable (PPV ~0.32 for both), so the problem is who gets flagged
in the first place.

## 2. Which definition of "fair"?

Every fairness principle compares the **same four numbers**, counted separately for girls and for boys:

| | prioritized (flagged) | not prioritized |
|---|---|---|
| **GAM** (became malnourished) | **A** = caught | **B** = missed |
| **no GAM** | **C** = false alarm | **D** = correctly left out |

Each principle picks one ratio of these numbers and asks that it be **the same for girls and boys**:

| Principle | The ratio that must be equal | Formula | In this clinic it means... | Fairlearn |
|---|---|---|---|---|
| **Demographic parity** | share prioritized | (A + C) / (A + B + C + D) | the same share of girls and boys get follow-up | `demographic_parity_difference` |
| **Equal opportunity** | sensitivity | A / (A + B) | a malnourished girl is as likely to be caught as a malnourished boy | `equal_opportunity_difference` |
| **Equalized odds** | sensitivity **and** false-alarm rate | A / (A + B) **and** C / (C + D) | as above, and a well girl is as likely to be called in by mistake as a well boy | `equalized_odds_difference` |
| **Predictive parity** | PPV | A / (A + C) | a flag means the same chance of GAM, whoever gets it | `MetricFrame` with `precision_score` |

### Practice 1

A clinic sees **100 girls and 100 boys**. 10 girls and 15 boys will develop GAM. The clinic prioritizes 20 girls and 20 boys:

| | girls: prioritized | girls: not | boys: prioritized | boys: not |
|---|---|---|---|---|
| **GAM** | 6 | 4 | 9 | 6 |
| **no GAM** | 14 | 76 | 11 | 74 |

> **PAUSE (5 min).** With pen and paper: compute the four ratios for girls and for boys. Which principles hold?

Then run the next cells to check your answers.

In [4]:
def show_2x2(cells):
    """The four numbers per group as one table: rows GAM / no GAM, columns prioritized / not."""
    columns = pd.MultiIndex.from_product([list(cells), ["prioritized", "not prioritized"]])
    rows = [[n for A, B, C, D in cells.values() for n in (A, B)],     # GAM: caught, missed
            [n for A, B, C, D in cells.values() for n in (C, D)]]     # no GAM: false alarm, correctly left out
    return pd.DataFrame(rows, index=["GAM", "no GAM"], columns=columns)


PRINCIPLES = {   # name: function of (A, B, C, D) -> (numerator, denominator, the sum written out)
    "Demographic parity: share prioritized = (A + C) / all":   lambda A, B, C, D: (A + C, A + B + C + D, f"({A} + {C}) / {A + B + C + D}"),
    "Equal opportunity: sensitivity = A / (A + B)":            lambda A, B, C, D: (A, A + B, f"{A} / ({A} + {B})"),
    "False-alarm rate = C / (C + D)":                          lambda A, B, C, D: (C, C + D, f"{C} / ({C} + {D})"),
    "Predictive parity: PPV = A / (A + C)":                    lambda A, B, C, D: (A, A + C, f"{A} / ({A} + {C})"),
}


def show_ratios(cells, digits=2, tol=0.01):
    """Each principle's ratio for every group, with the arithmetic written out, and whether they are equal."""
    rows = []
    for name, ratio in PRINCIPLES.items():
        row, values = {"principle and formula": name}, []
        for group, (A, B, C, D) in cells.items():
            num, den, written = ratio(A, B, C, D)
            row[group] = f"{written} = {num / den:.{digits}f}"
            values.append(num / den)
        row["equal?"] = "yes" if max(values) - min(values) <= tol else "no"
        rows.append(row)
    both = rows[1]["equal?"] == rows[2]["equal?"] == "yes"                        # sensitivity AND false-alarm rate
    rows.insert(3, {"principle and formula": "Equalized odds: both rows above equal", **{g: "" for g in cells},
                    "equal?": "yes" if both else "no"})
    return pd.DataFrame(rows).style.hide(axis="index").set_properties(**{"text-align": "left"})


practice_1 = {"girls": (6, 4, 14, 76), "boys": (9, 6, 11, 74)}     # (A, B, C, D) for each group
display(show_2x2(practice_1))
show_ratios(practice_1)

girls                        boys                
       prioritized not prioritized prioritized not prioritized
GAM              6               4           9               6
no GAM          14              76          11              74

principle and formula,girls,boys,equal?
Demographic parity: share prioritized = (A + C) / all,(6 + 14) / 100 = 0.20,(9 + 11) / 100 = 0.20,yes
Equal opportunity: sensitivity = A / (A + B),6 / (6 + 4) = 0.60,9 / (9 + 6) = 0.60,yes
False-alarm rate = C / (C + D),14 / (14 + 76) = 0.16,11 / (11 + 74) = 0.13,no
Equalized odds: both rows above equal,,,no
Predictive parity: PPV = A / (A + C),6 / (6 + 14) = 0.30,9 / (9 + 11) = 0.45,no


**Practice 1, answer.** The same share is prioritized (20%) and the same share of sick children is caught (60%):
**demographic parity and equal opportunity both hold.** But a flag for a girl is right 30% of the time, and for a boy
45%: **predictive parity fails**. The false-alarm rates differ too (16% vs 13%), so **equalized odds fails**.

### Practice 2

Another clinic, same children. This time it prioritizes 15 girls and 30 boys:

| | girls: prioritized | girls: not | boys: prioritized | boys: not |
|---|---|---|---|---|
| **GAM** | 5 | 5 | 10 | 5 |
| **no GAM** | 10 | 80 | 20 | 65 |

> **PAUSE (5 min).** Compute the four ratios again. Which principles hold now?

In [5]:
practice_2 = {"girls": (5, 5, 10, 80), "boys": (10, 5, 20, 65)}
display(show_2x2(practice_2))
show_ratios(practice_2)

girls                        boys                
       prioritized not prioritized prioritized not prioritized
GAM              5               5          10               5
no GAM          10              80          20              65

principle and formula,girls,boys,equal?
Demographic parity: share prioritized = (A + C) / all,(5 + 10) / 100 = 0.15,(10 + 20) / 100 = 0.30,no
Equal opportunity: sensitivity = A / (A + B),5 / (5 + 5) = 0.50,10 / (10 + 5) = 0.67,no
False-alarm rate = C / (C + D),10 / (10 + 80) = 0.11,20 / (20 + 65) = 0.24,no
Equalized odds: both rows above equal,,,no
Predictive parity: PPV = A / (A + C),5 / (5 + 10) = 0.33,10 / (10 + 20) = 0.33,yes


**Practice 2, answer.** Now a flag means the same thing for both (1 in 3 flagged children develops GAM):
**predictive parity holds**. Everything else fails: twice the share of boys is prioritized, and half of the sick girls
are missed against a third of the sick boys.

**What the two practices show.** Girls and boys here have **different GAM rates** (10% and 15%). When the rates
differ, no decision rule can make all four ratios equal at once. You have to **choose** which one matters most, and
that choice belongs to the people who run the programme, not to the data scientist.

### The real clinic, per 1,000 girls and 1,000 boys

The same tables for our model, scaled to 1,000 girls and 1,000 boys so the numbers are easy to read:

In [6]:
sex = evalset["sex"].to_numpy()


def four_cells(flags, per=1000):
    """(A, B, C, D) for girls and for boys, scaled to `per` children of each sex and rounded."""
    out = {}
    for g, name in [("F", "girls"), ("M", "boys")]:
        yy, ff = y[sex == g], flags[sex == g].astype(bool)
        counts = [((yy == 1) & ff).sum(), ((yy == 1) & ~ff).sum(), ((yy == 0) & ff).sum(), ((yy == 0) & ~ff).sum()]
        scaled = np.array(counts) * per / len(yy)
        whole = np.floor(scaled).astype(int)
        whole[np.argsort(whole - scaled)[: per - whole.sum()]] += 1     # round so the four add up to exactly `per`
        out[name] = tuple(int(n) for n in whole)
    return out


one_threshold = four_cells(flag)
display(show_2x2(one_threshold))
show_ratios(one_threshold, digits=3)

girls                        boys                
       prioritized not prioritized prioritized not prioritized
GAM             60              51          67              42
no GAM         130             759         142             749

principle and formula,girls,boys,equal?
Demographic parity: share prioritized = (A + C) / all,(60 + 130) / 1000 = 0.190,(67 + 142) / 1000 = 0.209,no
Equal opportunity: sensitivity = A / (A + B),60 / (60 + 51) = 0.541,67 / (67 + 42) = 0.615,no
False-alarm rate = C / (C + D),130 / (130 + 759) = 0.146,142 / (142 + 749) = 0.159,no
Equalized odds: both rows above equal,,,no
Predictive parity: PPV = A / (A + C),60 / (60 + 130) = 0.316,67 / (67 + 142) = 0.321,yes


- **Predictive parity holds** (PPV about 0.31 to 0.32): a flag means the same for a girl and a boy.
- **Equal opportunity does not hold**: of about 110 girls who develop GAM, the model catches 60; of about 110 boys, 67.
- **Demographic parity does not hold**, by a little: 19% of girls are prioritized, 21% of boys.
- So the model, as built, follows predictive parity. Nobody chose that: it is what a single threshold on a
  well-calibrated model gives you. Section 3 shows what the other choices would look like.

> **PAUSE (5–10 min). For girls and boys, which principle would you use, and why?** There is no right answer.
> Think about who pays for each choice: the sick child who is missed, the well child called in, the health worker.

### Regions

The same model and the same threshold, now by region. Read the four principles off the table:
the share prioritized (`alert_rate`), sensitivity, the false-alarm rate and PPV, next to each region's GAM rate
(`prevalence`).

In [7]:
from fairlearn.metrics import false_positive_rate

by_region = MetricFrame(metrics=metrics | {"false_alarm_rate": false_positive_rate},
                        y_true=y, y_pred=flag, sensitive_features=evalset["region"])
by_region.by_group.sort_values("prevalence")

,prevalence,alert_rate,sensitivity,miss_rate,ppv,n,false_alarm_rate
region,,,,,,,
Addis Ababa,0.026,0.021,0.154,0.846,0.186,2030.0,0.018
Oromia,0.099,0.165,0.496,0.504,0.296,7285.0,0.129
Amhara,0.109,0.195,0.497,0.503,0.278,5147.0,0.158
Sidama,0.111,0.153,0.509,0.491,0.368,1971.0,0.109
SNNP,0.114,0.214,0.613,0.387,0.326,3685.0,0.163
Tigray,0.134,0.268,0.685,0.315,0.343,1613.0,0.203
Afar,0.196,0.412,0.801,0.199,0.381,1076.0,0.317
Somali,0.212,0.487,0.794,0.206,0.346,1193.0,0.404


> **PAUSE (10 min). For regions, which principle would you use?** The same one you chose for girls and boys,
> or a different one? Why?
>
> Some questions to start the discussion:
> - Should every region get the same share of follow-up visits, or should it follow need?
> - Should a sick child in Addis Ababa have the same chance of being caught as a sick child in Afar?
> - Should a flag mean the same chance of GAM in every region?
> - Who should make this decision, and how would you explain it to a regional health office?

## 3. If we choose equal opportunity

Suppose the programme chooses equal opportunity for girls and boys. One direct way to get it: **separate thresholds
for girls and boys**, chosen so both groups get the same sensitivity, with the total number of flags kept within
capacity.

In [8]:
def summarise(name, scores, flags):
    # one row of the comparison table: overall quality, workload, and the girls/boys gap
    s = MetricFrame(metrics=true_positive_rate, y_true=y, y_pred=flags.astype(int),
                    sensitive_features=evalset["sex"]).by_group
    return {"approach": name, "roc_auc": roc_auc_score(y, scores), "alert_rate": flags.mean(),
            "sensitivity": flags[y == 1].mean(), "girls": s["F"], "boys": s["M"],
            "gap (boys-girls)": s["M"] - s["F"]}


top20 = lambda s: s >= np.quantile(s, 0.8)
results = [summarise("one threshold (as built)", score, top20(score))]


def equal_sensitivity_flags(scores, groups, capacity=0.20):
    """Per-group thresholds giving every group the same sensitivity, as high as capacity allows.

    Thresholds are chosen on known outcomes. In practice you set them on a past period
    and apply them going forward (exercise 2).
    """
    best = None
    for target in np.arange(0.40, 0.90, 0.005):              # try ever-higher sensitivity targets
        flags = np.zeros(len(scores), bool)
        for g in np.unique(groups):
            in_g = groups == g
            cases = np.sort(scores[in_g & (y == 1)])[::-1]   # risk scores of the group's GAM cases
            cut = cases[int(np.ceil(target * len(cases))) - 1]
            flags[in_g] = scores[in_g] >= cut
        if flags.mean() > capacity:
            break                                             # over capacity: keep the previous target
        best = flags
    return best


flags_eq = equal_sensitivity_flags(score, evalset["sex"].to_numpy())
results.append(summarise("per-sex thresholds", score, flags_eq))
pd.DataFrame(results).set_index("approach").round(3)

,roc_auc,alert_rate,sensitivity,girls,boys,gap (boys-girls)
approach,,,,,,
one threshold (as built),0.814,0.200,0.577,0.541,0.612,0.071
per-sex thresholds,0.814,0.198,0.580,0.581,0.580,-0.001


Equal sensitivity for girls and boys, within capacity, and overall sensitivity is no
worse. The model didn't change, only the decision rule on top of it.

In [9]:
rows = {}
for name, f in [("one threshold", flag), ("per-sex thresholds", flags_eq)]:
    t = MetricFrame(metrics={"alert_rate": selection_rate, "sensitivity": true_positive_rate, "ppv": precision_score},
                    y_true=y, y_pred=f.astype(int), sensitive_features=evalset["sex"]).by_group
    rows[name] = {f"{m} {s}": t.loc[s, m] for m in t.columns for s in ["F", "M"]}
pd.DataFrame(rows).T.round(3)

,alert_rate F,alert_rate M,sensitivity F,sensitivity M,ppv F,ppv M
one threshold,0.191,0.209,0.541,0.612,0.317,0.321
per-sex thresholds,0.210,0.186,0.581,0.580,0.308,0.341


Equal sensitivity (0.58 for both) has a price. More girls are flagged (21% vs 19% before) and fewer boys, and a flag
for a girl is now **less reliable** than one for a boy (PPV 0.31 vs 0.34). When a model scores two groups differently,
you cannot have equal sensitivity *and* equally reliable flags. Which one matters more is the question from the PAUSE.

### The three choices side by side, per 1,000 girls and 1,000 boys

Same model, same 20% capacity; only the decision rule changes:
- **one threshold** (as built): follows predictive parity;
- **per-sex thresholds** (above): equal opportunity;
- **top 20% of each sex**: demographic parity.

In [10]:
dp_flags = np.zeros(len(score), bool)
for g in ["F", "M"]:
    dp_flags[sex == g] = score[sex == g] >= np.quantile(score[sex == g], 0.8)     # top 20% within each sex

choices = {"one threshold": flag, "per-sex thresholds (equal opportunity)": flags_eq,
           "top 20% of each sex (demographic parity)": dp_flags}
for name, f in choices.items():
    print(f"\n{name}")
    cells = four_cells(f)
    display(show_2x2(cells))
    display(show_ratios(cells, digits=3))


one threshold


girls                        boys                
       prioritized not prioritized prioritized not prioritized
GAM             60              51          67              42
no GAM         130             759         142             749

principle and formula,girls,boys,equal?
Demographic parity: share prioritized = (A + C) / all,(60 + 130) / 1000 = 0.190,(67 + 142) / 1000 = 0.209,no
Equal opportunity: sensitivity = A / (A + B),60 / (60 + 51) = 0.541,67 / (67 + 42) = 0.615,no
False-alarm rate = C / (C + D),130 / (130 + 759) = 0.146,142 / (142 + 749) = 0.159,no
Equalized odds: both rows above equal,,,no
Predictive parity: PPV = A / (A + C),60 / (60 + 130) = 0.316,67 / (67 + 142) = 0.321,yes



per-sex thresholds (equal opportunity)


girls                        boys                
       prioritized not prioritized prioritized not prioritized
GAM             65              47          63              46
no GAM         145             743         123             768

principle and formula,girls,boys,equal?
Demographic parity: share prioritized = (A + C) / all,(65 + 145) / 1000 = 0.210,(63 + 123) / 1000 = 0.186,no
Equal opportunity: sensitivity = A / (A + B),65 / (65 + 47) = 0.580,63 / (63 + 46) = 0.578,yes
False-alarm rate = C / (C + D),145 / (145 + 743) = 0.163,123 / (123 + 768) = 0.138,no
Equalized odds: both rows above equal,,,no
Predictive parity: PPV = A / (A + C),65 / (65 + 145) = 0.310,63 / (63 + 123) = 0.339,no



top 20% of each sex (demographic parity)


girls                        boys                
       prioritized not prioritized prioritized not prioritized
GAM             63              49          66              43
no GAM         137             751         134             757

principle and formula,girls,boys,equal?
Demographic parity: share prioritized = (A + C) / all,(63 + 137) / 1000 = 0.200,(66 + 134) / 1000 = 0.200,yes
Equal opportunity: sensitivity = A / (A + B),63 / (63 + 49) = 0.562,66 / (66 + 43) = 0.606,no
False-alarm rate = C / (C + D),137 / (137 + 751) = 0.154,134 / (134 + 757) = 0.150,yes
Equalized odds: both rows above equal,,,no
Predictive parity: PPV = A / (A + C),63 / (63 + 137) = 0.315,66 / (66 + 134) = 0.330,no


In [11]:
summary = {}
for name, f in choices.items():
    c = four_cells(f)
    summary[name] = {"sick girls caught": c["girls"][0], "sick boys caught": c["boys"][0],
                     "all caught (of 2,000)": c["girls"][0] + c["boys"][0],
                     "false alarms, girls": c["girls"][2], "false alarms, boys": c["boys"][2]}
pd.DataFrame(summary).T

,sick girls caught,sick boys caught,"all caught (of 2,000)","false alarms, girls","false alarms, boys"
one threshold,60,67,127,130,142
per-sex thresholds (equal opportunity),65,63,128,145,123
top 20% of each sex (demographic parity),63,66,129,137,134


**Reading it.** All three rules catch about the same number of sick children in total (127 to 129 of about 220).
What changes is **who**:
- **One threshold:** 7 more sick boys caught than sick girls (67 vs 60). A flag means the same for both.
- **Per-sex thresholds:** sick girls and boys are caught at the same rate (58%). That is 5 more girls and 4 fewer
  boys caught, with 15 more false alarms among girls and 19 fewer among boys.
- **Top 20% of each sex:** the same workload for girls and boys. The gap in who is caught shrinks (56% vs 61%) but
  does not close.

No rule is free, and none is "the correct one". The data cannot decide this; people have to.

> **PAUSE (5 min).** You are the programme manager. Which row would you sign, and how would you explain it to a
> mother whose daughter was not called back? To a health worker who must do the extra visits?

### Fairlearn's general tool: `ThresholdOptimizer`

Fairlearn automates group-specific thresholds. Used with its defaults:

In [12]:
from fairlearn.postprocessing import ThresholdOptimizer

optimizer = ThresholdOptimizer(estimator=model, prefit=True, predict_method="predict_proba",
                               constraints="true_positive_rate_parity",   # = equal opportunity
                               objective="balanced_accuracy_score")
optimizer.fit(train[em.FEATURES], train[em.LABEL], sensitive_features=train["sex"])
flags_to = optimizer.predict(evalset[em.FEATURES], sensitive_features=evalset["sex"], random_state=0).astype(bool)
results.append(summarise("ThresholdOptimizer (defaults)", score, flags_to))
pd.DataFrame(results).set_index("approach").round(3)

,roc_auc,alert_rate,sensitivity,girls,boys,gap (boys-girls)
approach,,,,,,
one threshold (as built),0.814,0.200,0.577,0.541,0.612,0.071
per-sex thresholds,0.814,0.198,0.580,0.581,0.580,-0.001
ThresholdOptimizer (defaults),0.814,0.332,0.762,0.746,0.777,0.032


Sensitivity jumps, but **a third of all visits** are flagged: its objective (balanced
accuracy) knows nothing about the clinic's 20% capacity. The gap also shrinks less
than with our hand-made rule. A fairness tool optimises what you tell it to. Here the
real constraint is capacity, so the simple rule that encodes it does better.

Fairlearn also has **reductions** (e.g. `ExponentiatedGradient`) that build fairness
into training. They produce yes/no decisions rather than risk scores, so they fit
"approve / reject" settings better than "rank and follow up the top 20%". We tried
it; it isn't useful for this ranking setup.

## 4. Decide, document, monitor

| Rule | Principle it follows | Cost | Questions to ask |
|---|---|---|---|
| One threshold (as built) | predictive parity | none: it is the default | Is it acceptable that fewer sick girls are caught? |
| Per-sex thresholds | equal opportunity | uses sex at decision time; a flag for a girl is less reliable | Acceptable clinically? (Growth standards are already sex-specific.) |
| Top 20% of each sex | demographic parity | equal workload, not equal catching | Does equal workload matter more than equal catching? |

- **Write down the principle you chose**, the groups (sex, region, age band) and the tolerance (e.g. gap < 0.03),
  before looking at models.
- **Make it a gate.** The MLflow lesson's promotion rule already refuses a challenger that
  lowers girls' or boys' sensitivity; per-group metrics are logged with every run.
- **Monitor it.** Gaps can appear later, when the population or the measurement changes
  (Evidently lesson). Outcomes arrive ~90 days late, so fairness monitoring lags too.
- **The resume-matching use case** turns the questions around: protected attributes
  usually can't be used at decision time, and the harms are different. Same tools,
  different answers.